In [ ]:
%run ../shared_notebook_setup.py

In [ ]:
messages=[
    {
      "role": "user",
      "content": "How are you?",
    }
  ]

chat_completion = client.chat.completions.create(
  messages=messages,
  model=DATABRICKS_ENDPOINT
)

In [ ]:

print(chat_completion.choices[0].message.content)

In [7]:
def print_token_usage(chat_completion):
    print(f"Input tokens: {chat_completion.usage.prompt_tokens}")
    print(f"Output tokens: {chat_completion.usage.completion_tokens}")
    print(f"Total tokens: {chat_completion.usage.total_tokens}")

In [8]:
print_token_usage(chat_completion)

Input tokens: 14
Output tokens: 41
Total tokens: 55


In [10]:
DATABRICKS_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

chat_completion = client.chat.completions.create(
    messages=messages,
    model=DATABRICKS_ENDPOINT
)

print(chat_completion.choices[0].message.content)
print(f"Model used: {chat_completion.model}")
print_token_usage(chat_completion)

I'm just a language model, so I don't have feelings or emotions like humans do, but I'm here and ready to help you with any questions or topics you'd like to discuss. How about you? How's your day going so far?
Model used: meta-llama-3.3-70b-instruct-121024
Input tokens: 39
Output tokens: 52
Total tokens: 91


In [ ]:
usage = chat_completion.usage

print(f"Input tokens: {usage.prompt_tokens}")
print(f"Output tokens: {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")

if getattr(usage, "prompt_tokens_details", None):
    print("Input token details:", usage.prompt_tokens_details)

if getattr(usage, "completion_tokens_details", None):
    print("Output token details:", usage.completion_tokens_details)

In [11]:
## List all serving endpoints in the Databricks workspace
from urllib.request import Request, urlopen
import json

list_endpoints_url = f"{DATABRICKS_HOST}/api/2.0/serving-endpoints"
request = Request(list_endpoints_url, headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}"})

with urlopen(request) as response:
    payload = json.loads(response.read().decode("utf-8"))

endpoints = payload.get("endpoints", [])
print(f"Found {len(endpoints)} serving endpoints:")
for endpoint in endpoints:
    name = endpoint.get("name", "<unnamed>")
    config_name = endpoint.get("config", {}).get("served_entities", [{}])[0].get("entity_name", "")
    print(f"- {name}" + (f" (model: {config_name})" if config_name else ""))

Found 13 serving endpoints:
- databricks-claude-opus-4-8 (model: system.ai.databricks-claude-opus-4-8)
- databricks-gemini-3-5-flash (model: system.ai.databricks-gemini-3-5-flash)
- databricks-gpt-oss-120b (model: system.ai.gpt-oss-120b)
- databricks-gpt-oss-20b (model: system.ai.gpt-oss-20b)
- databricks-qwen3-next-80b-a3b-instruct (model: system.ai.qwen3-next-80b-a3b-instruct)
- databricks-qwen35-122b-a10b (model: system.ai.databricks-qwen35-122b-a10b)
- databricks-llama-4-maverick (model: system.ai.llama-4-maverick)
- databricks-gemma-3-12b (model: system.ai.gemma-3-12b-it)
- databricks-gte-large-en (model: system.ai.gte_large_en_v1_5)
- databricks-bge-large-en (model: system.ai.bge_large_en_v1_5)
- databricks-meta-llama-3-1-8b-instruct (model: system.ai.meta_llama_v3_1_8b_instruct)
- databricks-meta-llama-3-3-70b-instruct (model: system.ai.llama_v3_3_70b_instruct)
- databricks-qwen3-embedding-0-6b (model: system.ai.qwen3-embedding-0-6b)


In [12]:
import tiktoken

# Compare token IDs and token text across encodings
sample_text = "Isn't this so unbelievably amazing?"

# cl100k_base": "OpenAI BPE encoding used by many GPT-3.5/GPT-4-era models; good general-purpose tokenizer.",
# o200k_base": "Newer OpenAI BPE encoding with a larger vocabulary, used by newer model families for improved token efficiency.",

encodings = ["cl100k_base", "o200k_base"]
for encoding_name in encodings:
    encoding = tiktoken.get_encoding(encoding_name)
    ids = encoding.encode(sample_text)
    tokens = [encoding.decode([token_id]) for token_id in ids]

    print(f"\nEncoding: {encoding_name}")
    print(f"Text: {sample_text}")
    print(f"Token count: {len(ids)}")
    print("Token ID -> token text")
    for token_id, token_text in zip(ids, tokens):
        print(f"{token_id:>7} -> {repr(token_text)}")


Encoding: cl100k_base
Text: Isn't this so unbelievably amazing?
Token count: 8
Token ID -> token text
  89041 -> 'Isn'
    956 -> "'t"
    420 -> ' this'
    779 -> ' so'
  40037 -> ' unbelie'
  89234 -> 'vably'
   8056 -> ' amazing'
     30 -> '?'

Encoding: o200k_base
Text: Isn't this so unbelievably amazing?
Token count: 7
Token ID -> token text
   3031 -> 'Is'
   3023 -> "n't"
    495 -> ' this'
    813 -> ' so'
 180692 -> ' unbelievably'
   8467 -> ' amazing'
     30 -> '?'
